# 05 · Exploratory data analysis and model features

**Project context and attribution.** Developed for the IBM Applied Data Science Capstone (Coursera / IBM Skills Network). IBM supplied the original educational project framework and public teaching datasets; this is a reorganized, independently worded notebook incorporating the student's analysis and subsequent code corrections, with AI-assisted review. The archived teaching data are historical and are **not** a current account of SpaceX launches.

**Run environment:** Python 3 in VS Code's Jupyter notebook extension. Install the project dependencies outside the notebook using the repository's `requirements.txt`. There are no `pip`, `piplite`, JavaScript `fetch`, or notebook-environment-specific installation cells here.

## Aim and dataset

Explore how the **course-defined first-stage landing class** is associated with flight number, payload mass, launch site, orbit type, and calendar year. Then encode model features. This notebook works standalone with IBM's `dataset_part_2.csv`, or consumes the **local output of notebook 03** if available. No JavaScript fetch or browser-only package managers are used.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IBM_DATASET_PART_2 = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_2.csv"
local_input = OUTPUT_DIR / "dataset_part_2.csv"

if local_input.exists():
    df = pd.read_csv(local_input)
    print("Using notebook 03 output:", local_input)
else:
    df = pd.read_csv(IBM_DATASET_PART_2)
    print("Using IBM's frozen historical EDA dataset.")

needed = ["Class", "FlightNumber", "PayloadMass", "LaunchSite", "Orbit", "Date"]
missing = sorted(set(needed) - set(df.columns))
if missing:
    raise ValueError(f"EDA input is missing required columns: {missing}")

# 1 = recorded True outcome, 0 = other recorded outcomes (course definition)
df["Landing Outcome"] = df["Class"].map({0: "Failed / Other", 1: "Recorded Success"})
palette = {"Failed / Other": "#D04A42", "Recorded Success": "#228B60"}
print("Data shape:", df.shape)
df.head()

## Flight number versus payload mass

Color by landing class, noting that flight number also proxies changes in rocket hardware and operating practices over time.

In [ ]:
plt.figure(figsize=(11, 5))
sns.scatterplot(data=df, x="FlightNumber", y="PayloadMass", hue="Landing Outcome", palette=palette, s=68)
plt.xlabel("Flight Number")
plt.ylabel("Payload Mass (kg)")
plt.title("Payload mass across Falcon 9 flights")
plt.tight_layout()
plt.show()

## Task 1 · Flight number by launch site

In [ ]:
plt.figure(figsize=(11, 5))
sns.scatterplot(data=df, x="FlightNumber", y="LaunchSite", hue="Landing Outcome", palette=palette, s=65)
plt.xlabel("Flight Number")
plt.ylabel("Launch Site")
plt.title("Flight number by launch site and recorded landing outcome")
plt.tight_layout()
plt.show()

## Task 2 · Payload mass by launch site

In [ ]:
plt.figure(figsize=(11, 5))
sns.scatterplot(data=df, x="PayloadMass", y="LaunchSite", hue="Landing Outcome", palette=palette, s=65)
plt.xlabel("Payload Mass (kg)")
plt.ylabel("Launch Site")
plt.title("Payload mass and landing outcome across launch sites")
plt.tight_layout()
plt.show()

## Task 3 · Landing-success proportion by orbit

These are **descriptive within-sample rates**, not causal estimates. Small orbit categories can show 0% or 100% simply because there are few observations.

In [ ]:
orbit_success = (
    df.groupby("Orbit", dropna=False)["Class"]
      .agg(launches="size", landing_rate="mean")
      .sort_values("landing_rate", ascending=False)
      .reset_index()
)
print(orbit_success.to_string(index=False))
plt.figure(figsize=(9, 6))
sns.barplot(data=orbit_success, x="landing_rate", y="Orbit", color="#3478A5")
plt.xlabel("Recorded Landing Success Rate")
plt.ylabel("Orbit Type")
plt.xlim(0, 1)
plt.title("Landing success proportion by orbit")
plt.tight_layout()
plt.show()

## Task 4 · Flight number by orbit type

In [ ]:
plt.figure(figsize=(11, 6))
sns.scatterplot(data=df, x="FlightNumber", y="Orbit", hue="Landing Outcome", palette=palette, s=65)
plt.xlabel("Flight Number")
plt.ylabel("Orbit Type")
plt.title("Flight number and landing outcomes across orbits")
plt.tight_layout()
plt.show()

## Task 5 · Payload mass by orbit type

In [ ]:
plt.figure(figsize=(11, 6))
sns.scatterplot(data=df, x="PayloadMass", y="Orbit", hue="Landing Outcome", palette=palette, s=65)
plt.xlabel("Payload Mass (kg)")
plt.ylabel("Orbit Type")
plt.title("Payload mass and landing outcomes across orbits")
plt.tight_layout()
plt.show()

## Task 6 · Yearly first-stage landing trend

Extract year into a new column; do not overwrite the original date values.

In [ ]:
df["Year"] = pd.to_datetime(df["Date"], errors="coerce").dt.year
if df["Year"].isna().any():
    raise ValueError("Some Date entries cannot be parsed; inspect the input before graphing.")
yearly_success = (
    df.groupby("Year")["Class"].agg(launches="size", landing_rate="mean")
      .reset_index()
)
print(yearly_success.to_string(index=False))
plt.figure(figsize=(9, 5))
sns.lineplot(data=yearly_success, x="Year", y="landing_rate", marker="o")
plt.ylim(0, 1)
plt.xlabel("Year")
plt.ylabel("Recorded Landing Success Rate")
plt.title("First-stage landing success by year")
plt.tight_layout()
plt.show()

## Tasks 7–8 · Feature selection, one-hot encoding, and float conversion

Choose the same modeling features as the IBM exercise. The resulting file is **not a fitted scaler**. Train/test standardization should occur inside a scikit-learn pipeline, as done in the corrected classification notebook.

In [ ]:
feature_columns = [
    "FlightNumber", "PayloadMass", "Orbit", "LaunchSite", "Flights",
    "GridFins", "Reused", "Legs", "LandingPad", "Block",
    "ReusedCount", "Serial"
]
missing = sorted(set(feature_columns) - set(df.columns))
if missing:
    raise ValueError(f"Feature columns missing from dataset: {missing}")
features = df.loc[:, feature_columns].copy()
features_one_hot = pd.get_dummies(
    features, columns=["Orbit", "LaunchSite", "LandingPad", "Serial"],
    dummy_na=False
).astype("float64")
if features_one_hot.isna().any().any():
    raise ValueError("Feature matrix contains missing values; examine inputs before modeling.")
print("Feature matrix:", features_one_hot.shape)
features_one_hot.head()

## Save model-ready feature matrix

The exported matrix is **features only**; the separate `Class` column in `dataset_part_2.csv` is the target. Keep both datasets aligned if you rebuild an ML training workflow from these local outputs.

In [ ]:
output_file = OUTPUT_DIR / "dataset_part_3.csv"
features_one_hot.to_csv(output_file, index=False)
print("Saved:", output_file.resolve())
print("Rows:", len(features_one_hot), "Features:", features_one_hot.shape[1])
assert len(features_one_hot) == len(df)

## What this notebook demonstrates

- Categorical and numeric exploratory visualization
- Grouped outcome-rate summaries with sample sizes
- Temporal analysis without modifying dates
- Feature selection and one-hot encoding

**Interpretation reminder:** Course-defined `Class = 1` includes `True Ocean`, and patterns are associative rather than causal.

**Credit:** IBM Skills Network / Coursera Applied Data Science Capstone (original tasks and dataset); analysis wording, figures, and executable workflow reorganized for local VS Code use.